# EXPECT: phần thực hiện của Long

Notebook kiểm tra và hiển thị đầu ra thật đã lưu (không tự chạy huấn luyện hay đánh giá lại test khi Run All). Đặt RUN_TRAINING=True nếu muốn gọi lại pipeline dev.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd()
if not (ROOT / 'run_dev.py').exists():
    raise RuntimeError('Mở notebook từ thư mục Explainable_GEC_Long')
sys.path.insert(0, str(ROOT))
import bootstrap
from gec.safety import install_development_guard
install_development_guard()
import pandas as pd
OUT = ROOT / 'outputs/dev_run_001'
print('Python:', sys.version.split()[0])
print('Chế độ dev: test.json/test.pkl bị chặn trong phiên làm việc này')

Python: 3.12.14
Chế độ dev: test.json/test.pkl bị chặn trong phiên làm việc này


In [2]:
RUN_TRAINING = False
if RUN_TRAINING:
    import subprocess
    subprocess.run([sys.executable, '-u', 'run_dev.py', '--stage', 'all'], cwd=ROOT, check=True)
manifest = json.loads((OUT / 'run_manifest.json').read_text(encoding='utf-8'))
assert manifest['status'] == 'complete'
assert manifest['splits_read'] == ['train', 'dev']
assert not manifest['test_scored']
print('CLI run:', manifest['status'], manifest['completed_at'])

CLI run: complete 2026-10-22T20:52:41+07:00


In [3]:
from gec.data import load_development
train, dev, summaries = load_development(ROOT / 'data/json')
print(pd.DataFrame([{k:v for k,v in s.items() if k in ['split','records','tokens','SVA','VFORM','NNUM']} for s in summaries]).to_string(index=False))

split  records  tokens  SVA  VFORM  NNUM
train    14610  416571 1005   1190  1668
  dev     2367   68382   74     69   120


In [4]:
metrics = pd.read_csv(OUT / 'metrics_dev.csv')
print(metrics[metrics['label'].eq('MICRO')][['system','scoring','TP','FP','FN','P','R','F0.5']].to_string(index=False))

                system scoring  TP  FP  FN        P        R     F0.5
                 All_O   typed   0   0 263 0.000000 0.000000 0.000000
                 All_O untyped   0   0 263 0.000000 0.000000 0.000000
                   POS   typed  36 222 227 0.139535 0.136882 0.138996
                   POS untyped  37 221 226 0.143411 0.140684 0.142857
               LR_base   typed  38 112 225 0.253333 0.144487 0.220162
               LR_base untyped  40 110 223 0.266667 0.152091 0.231750
LR_reference_C3_tau065   typed  28  65 235 0.301075 0.106464 0.220472
LR_reference_C3_tau065 untyped  30  63 233 0.322581 0.114068 0.236220
             LR_parser   typed  27  55 236 0.329268 0.102662 0.228426
             LR_parser untyped  28  54 235 0.341463 0.106464 0.236887
                   kNN   typed  10  79 253 0.112360 0.038023 0.080775
                   kNN untyped  11  78 252 0.123596 0.041825 0.088853


In [5]:
grid = pd.read_csv(OUT / 'tuning_grid.csv')
assert len(grid) == 228
print(pd.read_csv(OUT / 'lr_best_per_C.csv')[['system','C','tau','F0.5','selected_C']].to_string(index=False))
print('Grid rows:', len(grid))

   system   C  tau     F0.5  selected_C
  LR_base 0.1 0.30 0.215054       False
  LR_base 0.3 0.30 0.207769       False
  LR_base 1.0 0.45 0.220162        True
  LR_base 3.0 0.65 0.220472       False
LR_parser 0.1 0.35 0.228426        True
LR_parser 0.3 0.35 0.228967       False
LR_parser 1.0 0.60 0.224525       False
LR_parser 3.0 0.50 0.210872       False
Grid rows: 228


In [6]:
classes = metrics[metrics['system'].isin(['LR_base','LR_parser']) & metrics['label'].ne('MICRO')]
print(classes[['system','label','TP','FP','FN','P','R','F0.5']].to_string(index=False))
fixed = pd.read_csv(ROOT / 'outputs/fixed_point_check/metrics_parser_fixed_point.csv')
print('\nFixed-point parser check (C=1.0, tau=0.45 vs tau=0.60):')
print(fixed[['system','scoring','label','TP','FP','FN','P','R','F0.5']].to_string(index=False))
cases = pd.read_csv(OUT / 'parser_four_cases.csv')
print('\nFour diagnostic cases:')
print(cases[['report_case_name','record_id','token','subject_text','reference_C3_tau065_prediction','base_prediction','parser_prediction']].to_string(index=False))

   system label  TP  FP  FN        P        R     F0.5
  LR_base   SVA  12  41  62 0.226415 0.162162 0.209790
  LR_base VFORM   7  17  62 0.291667 0.101449 0.212121
  LR_base  NNUM  19  54 101 0.260274 0.158333 0.230583
LR_parser   SVA  12  16  62 0.428571 0.162162 0.322581
LR_parser VFORM   5  15  64 0.250000 0.072464 0.167785
LR_parser  NNUM  10  24 110 0.294118 0.083333 0.195312

Fixed-point parser check (C=1.0, tau=0.45 vs tau=0.60):
                system scoring label  TP  FP  FN        P        R     F0.5
LR_parser_C1.0_tau0.45   typed   SVA  11  41  63 0.211538 0.148649 0.195035
LR_parser_C1.0_tau0.45   typed VFORM   8  37  61 0.177778 0.115942 0.160643
LR_parser_C1.0_tau0.45   typed  NNUM  19  44 101 0.301587 0.158333 0.255376
LR_parser_C1.0_tau0.45   typed MICRO  38 122 225 0.237500 0.144487 0.210410
LR_parser_C1.0_tau0.45 untyped MICRO  40 120 223 0.250000 0.152091 0.221484
 LR_parser_C1.0_tau0.6   typed   SVA   8  16  66 0.333333 0.108108 0.235294
 LR_parser_C1.0_tau0.6   t

In [7]:
# Trace 1 ID từ source -> đặc trưng -> xác suất/ngưỡng -> dự đoán -> gold (dev:001408, token 3 'realize')
row_fp02 = cases[cases['record_id'].eq('dev:001408')].iloc[0]
print('Record:', row_fp02['record_id'], '| Token index:', row_fp02['token_index'], '| Token:', row_fp02['token'], '| Gold:', row_fp02['gold'])
print('Source:', row_fp02['source'])
print('Parser subject:', row_fp02['subject_text'], '| tag:', row_fp02['subject_tag'], '| number:', row_fp02['subject_number'], '| distance:', row_fp02['subject_distance'])
print('LR_base (C=1.0, tau=0.45): p_error =', row_fp02['base_p_error'], '-> prediction =', row_fp02['base_prediction'])
print('LR_parser (C=0.1, tau=0.35): p_error =', row_fp02['parser_p_error'], '-> prediction =', row_fp02['parser_prediction'])
interv_dirs = sorted((ROOT / 'outputs').glob('fp02_intervention_dev_*'))
if interv_dirs:
    interv = pd.read_csv(interv_dirs[-1] / 'fp02_feature_intervention_dev.csv')
    print('\nFP02 intervention at C=1.0, tau=0.45:')
    print(interv[['scenario','subject_tag','subject_number','subject_distance','p_error','p_SVA','prediction']].to_string(index=False))

Record: dev:001408 | Token index: 3 | Token: realize | Gold: O
Source: Most people today realize that in our cities and in our planet there is an increasing problem of space and pollution .
Parser subject: people | tag: NNS | number: Plur | distance: -2
LR_base (C=1.0, tau=0.45): p_error = 0.9847113305447396 -> prediction = SVA
LR_parser (C=0.1, tau=0.35): p_error = 0.5022483954695685 -> prediction = SVA

FP02 intervention at C=1.0, tau=0.45:
     scenario subject_tag subject_number  subject_distance  p_error    p_SVA prediction
    as_parsed         NNS           Plur                -2 0.993020 0.987796        SVA
manual_people         NNS           Plur                -2 0.993020 0.987796        SVA
 manual_today          NN           Sing                -1 0.989731 0.978973        SVA


In [8]:
for person in ['Loan','Yen']:
    review = pd.read_csv(OUT / 'baseline_ref' / f'review_{person}.csv', keep_default_na=False)
    done = review['review_label_A_B_C'].str.strip().ne('').sum()
    print(person, 'review_fields_filled:', done, '/', len(review))
status = json.loads((OUT / 'baseline_ref' / 'review_status.json').read_text(encoding='utf-8'))
print('review_forms_filled:', status['review_forms_filled'], '| independent_reviews_complete:', status['independent_reviews_complete'], '| disagreements_before_discussion:', status['disagreements_before_discussion'])
print(f"A={status['A']}, B={status['B']} (in-scope 3 types={status['B_in_scope_3_types']}, out-of-scope={status['B_out_of_scope']}), C={status['C']}")
print(f"Adjusted precision (in-scope 3 types): {status['estimated_precision_in_scope_3_types']:.6f} | (all B): {status['estimated_precision']:.6f}")

Loan review_fields_filled: 63 / 63
Yen review_fields_filled: 63 / 63
review_forms_filled: True | independent_reviews_complete: False | disagreements_before_discussion: None
A=34, B=14 (in-scope 3 types=10, out-of-scope=4), C=15
Adjusted precision (in-scope 3 types): 0.430108 | (all B): 0.473118


In [9]:
verification = json.loads((OUT / 'verification.json').read_text(encoding='utf-8'))
assert verification['status'] == 'passed'
from gec.io_utils import sha256
locked = json.loads((OUT / 'config_locked.json').read_text(encoding='utf-8'))
for entry in locked['bundles'].values():
    assert sha256(ROOT / entry['path']) == entry['sha256']
print('Independent verification:', verification['status'])
print('Model checksums: matched')
print('Frozen dev lock state:', locked['status'])
test_dirs = sorted((ROOT / 'outputs').glob('final_test_*'))
if test_dirs:
    test_state = json.loads((test_dirs[-1] / 'evaluation_state.json').read_text(encoding='utf-8'))
    test_metrics = pd.read_csv(test_dirs[-1] / 'metrics_test.csv')
    print('\nSaved final test evaluation:', test_state['status'], '| completed_at:', test_state.get('completed_at'))
    print(test_metrics[test_metrics['label'].eq('MICRO')][['system','scoring','gold','TP','FP','FN','P','R','F0.5']].to_string(index=False))

Independent verification: passed
Model checksums: matched
Frozen dev lock state: dev_selection_frozen

Saved final test evaluation: complete | completed_at: 2026-11-03T13:19:38+07:00
   system scoring  gold  TP  FP  FN        P        R     F0.5
    All_O   typed   272   0   0 272 0.000000 0.000000 0.000000
    All_O untyped   272   0   0 272 0.000000 0.000000 0.000000
      POS   typed   272  31 217 241 0.125000 0.113971 0.122627
      POS untyped   272  32 216 240 0.129032 0.117647 0.126582
  LR_base   typed   272  41 154 231 0.210256 0.150735 0.194867
  LR_base untyped   272  43 152 229 0.220513 0.158088 0.204373
LR_parser   typed   272  27  79 245 0.254717 0.099265 0.193966
LR_parser untyped   272  27  79 245 0.254717 0.099265 0.193966
      kNN   typed   272  14  72 258 0.162791 0.051471 0.113636
      kNN untyped   272  14  72 258 0.162791 0.051471 0.113636


Kết luận: Trên dev, ở cùng C=1.0/tau=0.45, thêm 3 đặc trưng parser chưa giảm FP SVA (giữ 41 FP); ở cấu hình được chọn C=0.1/tau=0.35, SVA F0.5 tăng nhờ giảm FP (41 xuống 16), VFORM/NNUM giảm. Trên test (272 lỗi gold, chấm ngày 03/11/2026), LR_base (0.194867) và LR_parser (0.193966) đều vượt POS (0.122627) và kNN (0.113636); LR_parser có precision cao hơn và dẫn đầu ở SVA/VFORM, còn LR_base có recall cao hơn ở NNUM.